# Comparación de Modelos

Entrenamos 4 modelos: Regresión Logística, Random Forest, XGBoost y LightGBM.
Cada uno incluye manejo de desbalance de clases.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import roc_curve
import sys
sys.path.insert(0, '..')

from src.data.preprocess import dividir_datos
from src.models.train import obtener_modelos, entrenar_modelos, guardar_modelo
from src.models.evaluate import comparar_modelos, calcular_curva_aprobacion

df = pd.read_parquet('data/processed/dataset_features.parquet')
X_train, X_test, y_train, y_test = dividir_datos(df)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Tasa default train: {y_train.mean():.2%} | test: {y_test.mean():.2%}")

## Entrenamiento (puede tardar ~5-10 min)

In [ ]:
modelos = obtener_modelos()
modelos_entrenados = entrenar_modelos(X_train, y_train, modelos)
print("Entrenamiento completado.")

## Tabla comparativa de métricas

In [ ]:
tabla = comparar_modelos(modelos_entrenados, X_test, y_test)
print(tabla.to_string())
tabla

## Curvas ROC

In [ ]:
fig = go.Figure()
colores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for (nombre, modelo), color in zip(modelos_entrenados.items(), colores):
    y_proba = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = tabla.loc[nombre, 'auc_roc']
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, name=f"{nombre} (AUC={auc})", line=dict(color=color)
    ))
fig.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode='lines',
    line=dict(dash='dash', color='gray'), name='Azar'
))
fig.update_layout(
    title='Curvas ROC — Comparación de 4 Modelos',
    xaxis_title='Tasa de Falsos Positivos (FPR)',
    yaxis_title='Tasa de Verdaderos Positivos (TPR)',
    height=500
)
fig.show()

## Guardar modelo ganador

In [ ]:
mejor_nombre = tabla.index[0]
guardar_modelo(modelos_entrenados[mejor_nombre])
print(f"\nModelo ganador: {mejor_nombre}")
print(f"AUC-ROC: {tabla.loc[mejor_nombre, 'auc_roc']}")
print(f"KS: {tabla.loc[mejor_nombre, 'ks']}")
print(f"Gini: {tabla.loc[mejor_nombre, 'gini']}")